# Vuln2Action - evaluate T5 stage 1 checkpoint epoch 10

??nh gi? `checkpoint-6930` (epoch 10) c?a notebook
`minhiw/vuln2action-t5-stage1-paper`.

L? do: l?n train tr??c d?ng `load_best_model_at_end=True`, n?n `final/` b?
reload v? checkpoint epoch 1 v? eval loss th?p nh?t. Notebook n?y ??nh gi?
??ng checkpoint cu?i epoch 10.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

flow_hits = sorted(glob.glob("/kaggle/input/**/flow_validation.jsonl", recursive=True))
print("flow_validation hits:", flow_hits)
if not flow_hits:
    raise FileNotFoundError("flow_validation.jsonl not found")
FLOW_DIR = Path(flow_hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("FLOW_DIR:", FLOW_DIR)

ckpt_hits = sorted(Path("/kaggle/input").rglob("checkpoint-6930"))
print("checkpoint-6930 hits:", [str(p) for p in ckpt_hits])
if not ckpt_hits:
    ckpt_hits = sorted(Path("/kaggle/input").rglob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    print("fallback checkpoints:", [str(p) for p in ckpt_hits[-10:]])
if not ckpt_hits:
    raise FileNotFoundError("no checkpoint found")
CKPT = ckpt_hits[-1]
os.environ["CKPT"] = str(CKPT)
print("CKPT:", CKPT)
for name in ["config.json", "model.safetensors", "pytorch_model.bin", "tokenizer.json", "spiece.model"]:
    p = CKPT / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")


In [ ]:
!python scripts/evaluate_t5.py \
  --checkpoint "$CKPT" \
  --dataset "$FLOW_DIR" \
  --split validation \
  --target-field target_techniques \
  --output-dir /kaggle/working/t5_stage1_paper_eval10 \
  --max-input-length 256 \
  --max-target-length 256 \
  --batch-size 8 \
  --num-beams 4 \
  --fp16


In [ ]:
import json
from pathlib import Path
out = Path("/kaggle/working/t5_stage1_paper_eval10")
metrics = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\n--- sample predictions ---")
for i, line in enumerate((out / "predictions.jsonl").open(encoding="utf-8")):
    if i >= 5:
        break
    row = json.loads(line)
    print(f"\n[{i}] {row['cve_id']}")
    print("REF:", row["reference"][:300])
    print("PRED:", row["prediction"][:300])
